# RAG Nutrition - Quick Start

This notebook demonstrates how to use the RAG Nutrition system.

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Load Configuration

In [ ]:
from src.core.config import Config

config = Config.from_yaml('../configs/baseline.yaml')
print(f"Experiment: {config.experiment_name}")
print(f"Chunker: {config.chunker.type}")
print(f"Embedding: {config.embedding.model_name}")
print(f"Retriever: {config.retriever.type}")
print(f"LLM: {config.llm.model_name}")

## 2. Parse and Preprocess Documents

In [ ]:
from src.ingestion import PDFParser, TextPreprocessor

# Parse PDFs from data/raw
parser = PDFParser()
preprocessor = TextPreprocessor()

# Example: parse a single PDF
# doc = parser.parse('../data/raw/guia_alimentar.pdf')
# doc.content = preprocessor.preprocess(doc.content)
# print(f"Parsed: {doc.metadata['filename']}")
# print(f"Content length: {len(doc.content)} chars")

## 3. Create Pipeline and Ingest Documents

In [ ]:
from src.core.pipeline import RAGPipeline

# Create pipeline from config
pipeline = RAGPipeline.from_config('../configs/baseline.yaml')
print(pipeline)

In [ ]:
# Example: Ingest sample text
sample_text = """
O Guia Alimentar para a População Brasileira recomenda que a alimentação seja baseada 
em alimentos in natura ou minimamente processados. Esses alimentos devem ser a base 
da alimentação. Alimentos ultraprocessados devem ser evitados.

A água é essencial para a saúde. Recomenda-se beber pelo menos 2 litros de água por dia.
A quantidade pode variar conforme a atividade física e o clima.

Fibras são importantes para o funcionamento do intestino. Alimentos ricos em fibras 
incluem frutas, verduras, legumes e cereais integrais.
"""

chunks = pipeline.ingest(sample_text, metadata={"source": "sample"})
print(f"Created {len(chunks)} chunks")
for chunk in chunks:
    print(f"  - {chunk.id}: {chunk.content[:50]}...")

## 4. Query the System

In [ ]:
# Retrieve documents
query = "Quanta água devo beber por dia?"
docs = pipeline.retrieve(query)

print(f"Query: {query}")
print(f"\nRetrieved {len(docs)} documents:")
for doc in docs:
    print(f"  Score: {doc.score:.3f}")
    print(f"  Content: {doc.content[:100]}...")
    print()

In [ ]:
# Full RAG query (retrieve + generate)
response, sources = pipeline.query(query)

print(f"Query: {query}")
print(f"\nResponse:\n{response}")
print(f"\nSources used: {len(sources)}")

## 5. Compare Different Configurations

In [ ]:
# Load different configs and compare
configs_to_test = [
    '../configs/baseline.yaml',
    '../configs/experiment_hyde.yaml',
]

for config_path in configs_to_test:
    config = Config.from_yaml(config_path)
    print(f"\n=== {config.experiment_name} ===")
    print(f"Retriever: {config.retriever.type}")
    # Note: In real usage, you'd need to re-ingest documents for each config
    # or use the same vector store

## 6. Run Benchmark

In [ ]:
from src.evaluation import Benchmark

# Load benchmark with evaluation dataset
benchmark = Benchmark('../data/eval/golden_dataset.json')
print(f"Loaded {len(benchmark.questions)} evaluation questions")

# Run benchmark (uncomment when you have documents ingested)
# results = benchmark.run_config('../configs/baseline.yaml')
# for r in results:
#     print(f"{r.question_id}: P@1={r.metrics.precision_at_1:.2f}, MRR={r.metrics.mrr:.2f}")